# EXTERNAL iSign T5 — Kaggle Dataset-Level Evaluation (t5-small-motion ONLY)

Evaluation-only notebook. No training, no fine-tuning, no registry changes, no promotion, no frontend integration.

**Model:** `manavdhamecha77/iSign-t5-pose-to-text` checkpoint `t5-small-motion` ONLY (pose_dim=600, motion=True).
**Dataset:** held-out iSign test split (target 1,887 samples; actual verified count is recorded).
**Preprocessing:** verified external iSign contract (75 landmarks, +confidence 1.0, z-score axis=0 before motion, velocity concat, pad/truncate 500).
**Rule:** a prediction counts as correct ONLY if normalized prediction == normalized ground truth. Fluent English alone is never counted as correct.

**Pose access:** the 158+ GiB pose archive is NEVER fully downloaded. Only the required `.pose` members are retrieved via authenticated HTTP Range requests against the four-part ZIP (`iSign-poses_v1.1_part_aa..ad`, consecutive byte ranges of ONE ZIP64 archive) in `Exploration-Lab/iSign`. A manually uploaded pose dataset is NOT required.
**Calibration:** first run uses `MAX_EVAL_SAMPLES = 100`. Raise to 200, then to the full 1887 target, only after the calibration run passes.

Hard constraints (must hold):
- Do NOT modify the production SignSage inference pipeline.
- Do NOT train or fine-tune anything (`model.eval()` + `torch.no_grad()` only).
- Do NOT modify the model registry; do NOT set `SIGNSAGE_MODEL_PROMOTED=true`.
- Do NOT use `t5-base` / `t5-large` or any other checkpoint. ONLY `t5-small-motion`.
- Do NOT save hidden states, tensors, checkpoints, videos, images, dataset copies, or huge logs. Compact artifacts only (<10 MB).
- If the dataset cannot be accessed or its format is incompatible, STOP with an explicit blocker (no workaround, no invented numbers).
- If labels cannot be reliably mapped, STOP and document (no misleading accuracy).

Outputs (`evaluation_output/`): `predictions.csv`, `metrics.json`, `confusion_matrix.csv`, `evaluation_report.md`.

In [ ]:
import os
from pathlib import Path

# ---- All paths / knobs are configurable via environment; Kaggle defaults shown ----
CFG = {
    "MODEL_REPO": os.environ.get("ISIGN_MODEL_REPO", "manavdhamecha77/iSign-t5-pose-to-text"),
    "MODEL_VARIANT": os.environ.get("ISIGN_MODEL_VARIANT", "t5-small-motion"),
    "MODEL_DIR": os.environ.get("ISIGN_MODEL_DIR", ""),  # optional local/Kaggle-input checkpoint dir
    "BASE_TOKENIZER": os.environ.get("ISIGN_BASE_TOKENIZER", "t5-small"),
    "ISIGN_CSV": os.environ.get("ISIGN_CSV", ""),  # optional explicit CSV path
    "POSE_SOURCE": os.environ.get("ISIGN_POSE_SOURCE", "remote"),  # remote (default) | auto | local dir | local zip
    "REMOTE_CHUNK_MB": int(os.environ.get("REMOTE_CHUNK_MB", "4")),  # range-fetch granularity (MB)
    "HF_DATASET_REPO": os.environ.get("HF_DATASET_REPO", "Exploration-Lab/iSign"),
    "HF_CSV_FILE": os.environ.get("HF_CSV_FILE", "iSign_v1.1.csv"),
    "KAGGLE_INPUT_ROOT": os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"),
    "KAGGLE_WORKING": os.environ.get("KAGGLE_WORKING", "/kaggle/working"),
    "OUTPUT_DIR": os.environ.get("EVAL_OUTPUT_DIR", "/kaggle/working/evaluation_output"),
    # CALIBRATION DEFAULT: 100 samples first. Then 200, then 1887. Change via env MAX_EVAL_SAMPLES.
    "MAX_EVAL_SAMPLES": int(os.environ.get("MAX_EVAL_SAMPLES", "100")),
    "SAVE_EVERY": int(os.environ.get("SAVE_EVERY", "25")),  # incremental predictions write interval
    "MAX_FRAMES": 500,
    "POSE_DIM": 600,
    "USE_MOTION": True,
    "MAX_GEN_TOKENS": 128,
    "NUM_BEAMS": 4,
    "SPLIT_SEED": 42,
}
assert CFG["MODEL_VARIANT"] == "t5-small-motion", "ONLY t5-small-motion is allowed for this evaluation"
assert CFG["POSE_DIM"] == 600 and CFG["USE_MOTION"] is True, "t5-small-motion contract: pose_dim=600 + motion"
print("== Configuration summary ==")
print(f"Model: {CFG['MODEL_VARIANT']} (repo {CFG['MODEL_REPO']})")
print(f"Dataset: iSign held-out test set (requested {CFG['MAX_EVAL_SAMPLES']} this run; full target 1887; actual count recorded at runtime)")
print(f"Pose access: remote range-fetch only (chunk {CFG['REMOTE_CHUNK_MB']} MB); full archive is never downloaded")
print(f"Preprocessing: verified external iSign preprocessing (75 landmarks, z-score axis=0, motion, pad/truncate {CFG['MAX_FRAMES']})")
print(f"Generation: max_tokens={CFG['MAX_GEN_TOKENS']} num_beams={CFG['NUM_BEAMS']} (verified contract)")
print(f"Output dir: {CFG['OUTPUT_DIR']} (configurable via EVAL_OUTPUT_DIR)")


In [ ]:
import platform, sys
print(f"python={platform.python_version()} executable={sys.executable}")
try:
    import torch
    print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"gpu={torch.cuda.get_device_name(0)} count={torch.cuda.device_count()}")
    else:
        print("device fallback: CPU")
except Exception as e:
    print(f"torch import failed pre-install: {type(e).__name__}: {e}")
try:
    import transformers
    print(f"transformers={transformers.__version__}")
except Exception as e:
    print(f"transformers not yet installed: {type(e).__name__}")
try:
    import pose_format
    print("pose_format=available")
except Exception:
    print("pose_format=missing (will install)")
import pandas as _pd, numpy as _np
print(f"pandas={_pd.__version__} numpy={_np.__version__}")


In [ ]:
import subprocess, sys
# Minimal eval-only dependencies. torch/numpy/pandas are preinstalled on Kaggle images.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "huggingface_hub", "pose-format", "pandas", "requests"], check=True)
import torch, transformers
print(f"installed: torch={torch.__version__} transformers={transformers.__version__}")


In [ ]:
import random
import numpy as np, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} (GPU if available, otherwise CPU)")
torch.set_grad_enabled(False)


## Verified external preprocessing contract (read-only, eval-only)

Sources: `docs/EXTERNAL_EVALUATION.md` (§§1–2) + `docs/reference/contract-verification.md`,
`backend/app/ml/adapters/i_sign_t5_adapter.py` (`signage_to_external`), external `inference/pose_utils.py` + `inference/model.py`.

| Step | Exact rule |
|---|---| 
| 1. Component select | SignSage 543 order POSE(0:33)+FACE(33:501)+LEFT(501:522)+RIGHT(522:543); keep POSE + LEFT + RIGHT = 75 landmarks; discard FACE |
| 2. Confidence | append 1.0 per landmark -> 75 x 4 = 300 values/frame (`include_confidence=True` fallback) |
| 3. Normalize | z-score across axis=0: `(x - nanmean(x,0)) / where(nanstd==0, 1.0, nanstd)`, then `nan_to_num` |
| 4. Motion | FROM normalized features: `velocity[0]=0`, `velocity[t]=pose[t]-pose[t-1]`; concat `[pose \| velocity]` -> 600/frame |
| 5. Sequence | pad with zeros / truncate to exactly 500 frames; `actual_length = min(T, 500)` |
| 6. Tensors | `pose_tensor=(1,500,600)` float32, `pose_length=[actual_length]` long; attention mask from `actual_length` |

Order matters: normalize BEFORE motion, motion BEFORE frame padding. No SignSage hip-centering here.


In [ ]:
import numpy as np

TOTAL_LANDMARKS = 543
POSE_SL = slice(0, 33)
LEFT_SL = slice(501, 522)
RIGHT_SL = slice(522, 543)
SPATIAL_DIM, MOTION_DIM, MAX_FRAMES = 300, 600, 500

def normalize_text(text) -> str:
    """Lowercase + collapse whitespace (same as research/training/audit_dataset.py)."""
    return " ".join(str(text).lower().strip().split())

def signage_to_external_verified(landmarks_543: np.ndarray, use_motion: bool = True, max_frames: int = 500):
    """Verbatim port of backend/app/ml/adapters/i_sign_t5_adapter.py::signage_to_external.
    Input (T,543,3) -> (max_frames,300|600) float32 + actual_length. Eval-only, no training."""
    if not isinstance(landmarks_543, np.ndarray):
        raise TypeError("Expected numpy.ndarray (T,543,3)")
    if landmarks_543.ndim != 3 or landmarks_543.shape[1] != 543 or landmarks_543.shape[2] != 3:
        raise ValueError(f"Expected (T,543,3), got {landmarks_543.shape} (incompatible representation -- refusing)")
    T = int(landmarks_543.shape[0])
    if T < 1:
        raise ValueError("Empty pose sequence (incompatible -- refusing)")
    pose_33 = landmarks_543[:, POSE_SL, :]
    left_21 = landmarks_543[:, LEFT_SL, :]
    right_21 = landmarks_543[:, RIGHT_SL, :]
    parts = []
    for comp in (pose_33, left_21, right_21):
        flat3 = comp.reshape(T, -1, 3).astype(np.float32)
        conf = np.ones((T, flat3.shape[1], 1), dtype=np.float32)  # documented fallback 1.0
        parts.append(np.concatenate([flat3, conf], axis=2).reshape(T, -1))
    combined = np.concatenate(parts, axis=1).astype(np.float32)  # (T,300)
    assert combined.shape[1] == SPATIAL_DIM, combined.shape
    mean = np.nanmean(combined, axis=0, keepdims=True)
    std = np.nanstd(combined, axis=0, keepdims=True)
    combined = (combined - mean) / np.where(std == 0, 1.0, std)
    combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    if use_motion:
        velocity = np.zeros_like(combined, dtype=np.float32)
        velocity[1:] = combined[1:] - combined[:-1]
        combined = np.concatenate([combined, velocity], axis=1).astype(np.float32)  # (T,600)
        assert combined.shape[1] == MOTION_DIM, combined.shape
    if T >= max_frames:
        return combined[:max_frames].copy(), max_frames
    padded = np.zeros((max_frames, combined.shape[1]), dtype=np.float32)
    padded[:T] = combined
    return padded, T

def read_pose_to_543(raw_bytes: bytes, uid: str) -> np.ndarray:
    """Raw .pose bytes -> (T,543,3) float32. Verifies component layout before positional slicing."""
    from pose_format import Pose
    pose = Pose.read(raw_bytes)
    names = [c.name for c in pose.header.components]
    expected_prefix = ["POSE_LANDMARKS", "FACE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS"]
    if names[:4] != expected_prefix:
        raise ValueError(f"Pose header {names} for {uid} does not start with {expected_prefix} -- refusing to slice positionally")
    total = sum(len(c.points) for c in pose.header.components)
    if total < TOTAL_LANDMARKS:
        raise ValueError(f"Pose header has only {total} points for {uid}, need {TOTAL_LANDMARKS}")
    data = np.asarray(np.ma.filled(pose.body.data, 0.0), dtype=np.float32)
    if data.ndim != 4 or data.shape[1] < 1:
        raise ValueError(f"Unexpected pose tensor {data.shape} for {uid}")
    return data[:, 0, :TOTAL_LANDMARKS, :3].astype(np.float32)  # (T,543,3)

# Self-check on synthetic data (shapes + motion formula only; NOT an accuracy claim).
_rng = np.random.default_rng(0)
_p, _l = signage_to_external_verified(_rng.random((7, 543, 3), dtype=np.float32), use_motion=True, max_frames=500)
assert _p.shape == (500, 600) and _l == 7 and np.isfinite(_p).all()
print("preprocessing self-check OK: (7,543,3) -> (500,600), length=7")


In [ ]:
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from huggingface_hub import snapshot_download
from pathlib import Path

class PoseEncoder(torch.nn.Module):
    """Verified arch from inference/model.py: Linear(pose_dim->256)->ReLU->Dropout(0.1)->Linear(256->d_model)."""
    def __init__(self, pose_dim: int, embedding_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(pose_dim, hidden_dim), torch.nn.ReLU(),
            torch.nn.Dropout(0.1), torch.nn.Linear(hidden_dim, embedding_dim))
    def forward(self, pose):
        return self.encoder(pose)

def resolve_checkpoint():
    if CFG["MODEL_DIR"] and Path(CFG["MODEL_DIR"]).is_dir():
        ckpt = Path(CFG["MODEL_DIR"])
        print(f"using local checkpoint dir: {ckpt}")
    else:
        print(f"downloading {CFG['MODEL_REPO']} subfolder {CFG['MODEL_VARIANT']} (compact: t5-small-motion only)")
        root = snapshot_download(repo_id=CFG["MODEL_REPO"], allow_patterns=[f"{CFG['MODEL_VARIANT']}/*"])
        ckpt = Path(root) / CFG["MODEL_VARIANT"]
    t5_dir, pe_path = ckpt / "t5", ckpt / "pose_encoder.pt"
    if not t5_dir.is_dir() or not pe_path.is_file():
        raise RuntimeError(f"checkpoint incomplete at {ckpt}: need t5/ + pose_encoder.pt")
    return ckpt, t5_dir, pe_path

CKPT_DIR, T5_DIR, PE_PATH = resolve_checkpoint()
print(f"checkpoint: {CFG['MODEL_REPO']}:{CFG['MODEL_VARIANT']} @ {CKPT_DIR}")
tokenizer = AutoTokenizer.from_pretrained(CFG["BASE_TOKENIZER"])
t5_model = T5ForConditionalGeneration.from_pretrained(str(T5_DIR)).to(DEVICE)
t5_model.eval()
pose_encoder = PoseEncoder(pose_dim=600, embedding_dim=t5_model.config.d_model).to(DEVICE)
pose_encoder.load_state_dict(torch.load(str(PE_PATH), map_location=DEVICE))
pose_encoder.eval()
assert t5_model.training is False and pose_encoder.training is False
print(f"loaded t5-small-motion: d_model={t5_model.config.d_model} device={DEVICE} (eval-only, no training)")
MODEL_OK = True


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

def clean_dataframe_lite(df: pd.DataFrame):
    """Minimal port of research/training/audit_dataset.py::clean_dataframe (uid/text only)."""
    df = df.copy()
    df["uid"] = df["uid"].astype(str).str.strip()
    df["text"] = df["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    parts = df["uid"].str.rsplit("-", n=1, expand=True)
    df["video_id"] = parts[0].where(parts[1].notna() & (parts[0].str.strip() != ""), None)
    df = df.loc[df["video_id"].notna()].copy()
    df["text_norm"] = df["text"].map(normalize_text)
    junk = df["text_norm"].eq("") | df["uid"].eq("")
    return df.loc[~junk].reset_index(drop=True)

def video_split_lite(df: pd.DataFrame, seed: int = 42):
    """Leak-safe 80/10/10 split by video_id (same policy as audit_dataset.video_split)."""
    vids = np.array(sorted(df["video_id"].dropna().unique()), dtype=object)
    rng = np.random.default_rng(seed)
    rng.shuffle(vids)
    n = len(vids)
    c1, c2 = int(0.80 * n), int(0.90 * n)
    tr_ids, va_ids, te_ids = set(vids[:c1]), set(vids[c1:c2]), set(vids[c2:])
    te = df[df["video_id"].isin(te_ids)].reset_index(drop=True)
    return te

def find_csv():
    if CFG["ISIGN_CSV"] and Path(CFG["ISIGN_CSV"]).is_file():
        return CFG["ISIGN_CSV"]
    for pat in ("**/iSign_v1.1.csv", "**/isign*.csv", "**/*.csv"):
        hits = sorted(Path(CFG["KAGGLE_INPUT_ROOT"]).glob(pat)) if Path(CFG["KAGGLE_INPUT_ROOT"]).is_dir() else []
        for h in hits:
            try:
                if {"uid", "text"} <= set(pd.read_csv(h, nrows=0).columns):
                    print(f"found CSV under Kaggle input: {h}")
                    return str(h)
            except Exception:
                continue
    # Small metadata CSV fallback from the gated HF dataset (needs HF_TOKEN secret).
    import os
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = None
    if not tok:
        raise RuntimeError("BLOCKER: no iSign CSV found (set ISIGN_CSV or attach it under /kaggle/input) and no HF_TOKEN secret for HF fallback. Stopping -- no workaround.")
    from huggingface_hub import HfApi, hf_hub_download
    cands = [f for f in HfApi(token=tok).list_repo_files(CFG["HF_DATASET_REPO"], repo_type="dataset") if f.endswith(".csv")]
    want = CFG["HF_CSV_FILE"].lower()
    pick = next((f for f in cands if f.lower() == want or f.lower().endswith("/" + want)), None)
    if pick is None:
        raise RuntimeError(f"BLOCKER: cannot choose iSign CSV on HF; candidates={cands}. Stopping.")
    path = hf_hub_download(repo_id=CFG["HF_DATASET_REPO"], filename=pick, repo_type="dataset", token=tok)
    print(f"downloaded CSV {pick}")
    return path

CSV_PATH = find_csv()
df_all = pd.read_csv(CSV_PATH)
if not {"uid", "text"} <= set(df_all.columns):
    raise RuntimeError(f"BLOCKER: CSV {CSV_PATH} header {list(df_all.columns)} missing uid/text (incompatible -- stopping).")
df_clean = clean_dataframe_lite(df_all)
df_test_full = video_split_lite(df_clean, seed=CFG["SPLIT_SEED"])
N_TARGET = min(int(CFG["MAX_EVAL_SAMPLES"]), 1887, len(df_test_full))
df_test = df_test_full.iloc[:N_TARGET].reset_index(drop=True)
print(f"CSV: {CSV_PATH} rows={len(df_all)} usable={len(df_clean)} test_pool={len(df_test_full)} evaluating={len(df_test)}")
if len(df_test) == 0:
    raise RuntimeError("BLOCKER: held-out test split is empty after cleaning (stopping, no invented labels).")
CSV_OK = True


In [ ]:
# Pose source: remote range-streaming (default) or a local dir/zip override. No full download, ever.
# Remote classes are an eval-only port of research/training/train_sign_to_text.py:
# RemoteConcatFile + RemotePoseSource (authenticated HTTP Range, CDN re-resolve, 206/Content-Range
# validation, retries, bounded chunk cache). Preprocessing contract is untouched (see adapter cell).
import io, os, re, time, zipfile
from pathlib import Path
import numpy as np

POSE_PARTS = ["iSign-poses_v1.1_part_aa", "iSign-poses_v1.1_part_ab",
              "iSign-poses_v1.1_part_ac", "iSign-poses_v1.1_part_ad"]
POSE_INNER_PREFIX = "iSign-poses_v1.1/"

def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN") or None
    except Exception:
        return None

class DirPoseSource:
    def __init__(self, root): self.root = Path(root)
    def read(self, uid):
        p = self.root / f"{uid}.pose"
        if not p.is_file(): raise FileNotFoundError(uid)
        return p.read_bytes()

class ZipPoseSource:
    def __init__(self, path):
        self.zf = zipfile.ZipFile(path, allowZip64=True); self.names = set(self.zf.namelist())
    def read(self, uid):
        for cand in (f"{POSE_INNER_PREFIX}{uid}.pose", f"{uid}.pose"):
            if cand in self.names:
                with self.zf.open(cand) as fh: return fh.read()
        raise FileNotFoundError(uid)

class RemoteConcatFile(io.RawIOBase):
    """Seekable view over the 4-part pose ZIP: parts are consecutive byte ranges of ONE ZIP64
    archive, so zipfile reads the central directory once, then fetches only requested members
    via bounded HTTP ranges. Only fetched chunks reside in memory (at most 8 cached)."""
    def __init__(self, token: str, chunk_mb: int = 4):
        import requests
        from huggingface_hub import hf_hub_url
        super().__init__()
        self._requests = requests
        self.urls = [hf_hub_url(CFG["HF_DATASET_REPO"], p, repo_type="dataset") for p in POSE_PARTS]
        self.sizes = [self._probe(u, token) for u in self.urls]
        self.starts = np.cumsum([0] + self.sizes[:-1]).tolist()
        self.total_size = int(sum(self.sizes))
        self.pos = 0
        self.chunk_size = int(chunk_mb * 1024 * 1024)
        self.session = requests.Session()
        self.session.headers.update({"Authorization": f"Bearer {token}", "User-Agent": "SignSage-eval/1.0"})
        from collections import OrderedDict
        self.cache: OrderedDict = OrderedDict()
        self.resolved: dict = {}
    def _probe(self, url: str, token: str) -> int:
        r = self._requests.get(url, headers={"Authorization": f"Bearer {token}", "Range": "bytes=0-0",
                                             "User-Agent": "SignSage-eval/1.0"},
                               timeout=(20, 60), allow_redirects=True)
        if r.status_code != 206:
            raise RuntimeError(f"Could not probe remote part size: HTTP {r.status_code}")
        total = r.headers.get("Content-Range", "0/0").rsplit("/", 1)[-1]
        if not total.isdigit():
            raise RuntimeError(f"Could not parse total size from Content-Range: {r.headers.get('Content-Range')!r}")
        return int(total)
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, offset, whence=os.SEEK_SET):
        new = offset if whence == os.SEEK_SET else (self.pos + offset if whence == os.SEEK_CUR else self.total_size + offset)
        if new < 0: raise ValueError("negative seek position")
        self.pos = min(int(new), self.total_size)
        return self.pos
    def _part_for(self, pos):
        idx = max(0, min(int(np.searchsorted(self.starts, pos, side="right") - 1), len(self.urls) - 1))
        return idx, pos - self.starts[idx]
    def _fetch(self, part_idx, local_start, n):
        key = (part_idx, (local_start // self.chunk_size) * self.chunk_size)
        if key in self.cache:
            self.cache.move_to_end(key)
            returned_start, content = self.cache[key]
            piece = content[local_start - returned_start:local_start - returned_start + n]
            if len(piece) == n: return piece
            self.cache.pop(key, None)
        chunk_start = key[1]
        chunk_end = min(self.sizes[part_idx] - 1, chunk_start + self.chunk_size - 1)
        last = None
        for attempt in range(5):
            try:
                request_url = self.resolved.get(part_idx, self.urls[part_idx])
                r = self.session.get(request_url, headers={"Range": f"bytes={chunk_start}-{chunk_end}"},
                                     timeout=(20, 180), allow_redirects=True)
                if r.status_code != 206 and request_url != self.urls[part_idx]:
                    self.resolved.pop(part_idx, None)
                    r = self.session.get(self.urls[part_idx], headers={"Range": f"bytes={chunk_start}-{chunk_end}"},
                                         timeout=(20, 180), allow_redirects=True)
                if r.status_code != 206:
                    raise RuntimeError(f"range request failed: HTTP {r.status_code}")
                m = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", r.headers.get("Content-Range", "").strip())
                if not m: raise RuntimeError(f"Invalid Content-Range: {r.headers.get('Content-Range')!r}")
                rs, re_, total = map(int, m.groups())
                if total != self.sizes[part_idx]: raise RuntimeError(f"part {part_idx} size changed")
                if not (rs <= chunk_start and chunk_end <= re_): raise RuntimeError("range does not cover request")
                if len(r.content) != re_ - rs + 1: raise RuntimeError(f"incomplete range body for part {part_idx}")
                self.resolved[part_idx] = r.url
                self.cache[key] = (rs, r.content)
                self.cache.move_to_end(key)
                while len(self.cache) > 8: self.cache.popitem(last=False)
                piece = r.content[local_start - rs:local_start - rs + n]
                if len(piece) != n: raise RuntimeError(f"short range slice for part {part_idx}")
                return piece
            except (self._requests.exceptions.Timeout, self._requests.exceptions.ConnectionError,
                    self._requests.exceptions.ChunkedEncodingError) as exc:
                last = exc
                self.resolved.pop(part_idx, None)
                time.sleep(1.5 * 2 ** attempt)
        raise ConnectionError(f"range fetch failed after retries: {key}") from last
    def read(self, n=-1):
        if self.pos >= self.total_size: return b""
        if n is None or n < 0: n = self.total_size - self.pos
        n = min(int(n), self.total_size - self.pos)
        out = bytearray()
        remaining = n
        while remaining:
            part_idx, local = self._part_for(self.pos)
            take = min(remaining, self.sizes[part_idx] - local, self.chunk_size - (local % self.chunk_size))
            out.extend(self._fetch(part_idx, local, take))
            self.pos += take
            remaining -= take
        return bytes(out)

class RemotePoseSource:
    def __init__(self, token: str, chunk_mb: int = 4):
        self.zf = zipfile.ZipFile(RemoteConcatFile(token, chunk_mb), mode="r", allowZip64=True)
        self.names = set(self.zf.namelist())
    def read(self, uid):
        for cand in (f"{POSE_INNER_PREFIX}{uid}.pose", f"{uid}.pose"):
            if cand in self.names:
                with self.zf.open(cand) as fh: return fh.read()
        raise FileNotFoundError(uid)

HF_TOKEN = get_hf_token()  # never printed; Bearer header only
_src = CFG["POSE_SOURCE"]
if _src in ("remote", "auto"):
    _local = None
    _root = Path(CFG["KAGGLE_INPUT_ROOT"])
    if _root.is_dir():
        for z in sorted(_root.rglob("*.zip")):
            if zipfile.is_zipfile(z):
                _local = ("zip", ZipPoseSource(str(z))); print(f"local pose zip: {z}"); break
        if _local is None:
            _dirs = [d for d in _root.rglob("*") if d.is_dir() and next(d.glob("*.pose"), None)]
            if _dirs:
                _local = ("dir", DirPoseSource(str(_dirs[0]))); print(f"local pose dir: {_dirs[0]}")
    if _local is not None and _src == "auto":
        POSE_KIND, POSE_STORE = _local
    else:
        if not HF_TOKEN:
            raise RuntimeError("BLOCKER: remote pose streaming needs HF_TOKEN (Kaggle secret or env). Stopping -- no workaround.")
        POSE_STORE = RemotePoseSource(HF_TOKEN, chunk_mb=int(CFG["REMOTE_CHUNK_MB"]))
        POSE_KIND = "remote"
        print(f"remote pose source: {CFG['HF_DATASET_REPO']} parts={len(POSE_PARTS)} chunk={CFG['REMOTE_CHUNK_MB']} MB")
elif zipfile.is_zipfile(_src):
    POSE_KIND, POSE_STORE = "zip", ZipPoseSource(_src)
elif Path(_src).is_dir():
    POSE_KIND, POSE_STORE = "dir", DirPoseSource(_src)
else:
    raise RuntimeError(f"BLOCKER: ISIGN_POSE_SOURCE={_src!r} is not remote/auto/dir/zip (stopping).")
print(f"pose source kind: {POSE_KIND} (full archive is never downloaded)")


## Preflight gate (runs before any evaluation)

Ten ordered checks: token → repo access → CSV → test pool → pose source → member fetch →
parse → shape → adapter → single-sample inference. Any failure raises and stops the notebook
with a diagnostic; the evaluation loop never runs on a broken pipeline.

In [ ]:
import torch
from huggingface_hub import HfApi

@torch.no_grad()
def generate_text(pose_600: torch.Tensor, actual_len: int) -> str:
    pose_600 = pose_600.to(DEVICE)
    emb = pose_encoder(pose_600)  # (1,500,512)
    L = emb.size(1)
    mask = (torch.arange(L, device=DEVICE)[None, :] < torch.tensor([actual_len], device=DEVICE)[:, None]).long()
    gen = t5_model.generate(inputs_embeds=emb, attention_mask=mask, max_length=CFG["MAX_GEN_TOKENS"],
                            num_beams=CFG["NUM_BEAMS"], early_stopping=True, length_penalty=2.0)
    return tokenizer.decode(gen[0], skip_special_tokens=True)

print("== Preflight ==")
if not HF_TOKEN:
    raise RuntimeError("PREFLIGHT FAIL [1/10 HF_TOKEN]: no token (Kaggle secret HF_TOKEN or env). Stopping.")
print("[1/10] HF_TOKEN: PRESENT (value never printed)")
try:
    _repo_files = set(HfApi(token=HF_TOKEN).list_repo_files(CFG["HF_DATASET_REPO"], repo_type="dataset"))
except Exception as e:
    raise RuntimeError(f"PREFLIGHT FAIL [2/10 repo access]: {type(e).__name__}: {e}. Stopping.")
_missing = [p for p in POSE_PARTS + [CFG["HF_CSV_FILE"]] if p not in _repo_files]
if _missing:
    raise RuntimeError(f"PREFLIGHT FAIL [2/10 repo access]: missing {_missing}. Stopping.")
print(f"[2/10] repo access: OK ({CFG['HF_DATASET_REPO']}, 4 pose parts + CSV listed)")
print(f"[3/10] CSV: OK ({CSV_PATH}, usable={len(df_clean)})")
if len(df_test) == 0:
    raise RuntimeError("PREFLIGHT FAIL [4/10 test pool]: empty selection. Stopping.")
print(f"[4/10] test pool: OK (pool={len(df_test_full)}, selected={len(df_test)})")
print(f"[5/10] pose source: OK ({POSE_KIND})")
_probe_uid = str(df_test["uid"].iloc[0])
try:
    _raw = POSE_STORE.read(_probe_uid)
except Exception as e:
    raise RuntimeError(f"PREFLIGHT FAIL [6/10 member fetch] uid={_probe_uid}: {type(e).__name__}: {e}. Stopping.")
print(f"[6/10] member fetch: OK (uid={_probe_uid}, bytes={len(_raw)})")
_arr = read_pose_to_543(_raw, _probe_uid)
print(f"[7/10] pose parse: OK (uid={_probe_uid})")
if _arr.ndim != 3 or _arr.shape[1:] != (543, 3):
    raise RuntimeError(f"PREFLIGHT FAIL [8/10 pose shape]: {_arr.shape}. Stopping.")
print(f"[8/10] pose shape: OK ({_arr.shape})")
_pre, _plen = signage_to_external_verified(_arr, use_motion=True, max_frames=500)
assert _pre.shape == (500, 600) and np.isfinite(_pre).all()
print("[9/10] adapter: OK ((500,600) float32, finite)")
_txt = generate_text(torch.from_numpy(_pre).unsqueeze(0).float(), int(_plen))
print(f"[10/10] single-sample inference: OK ({len(_txt)} chars)")
PREFLIGHT_OK = True
SINGLE_INFER_OK = True
print("PREFLIGHT: PASS")


In [ ]:
import time, torch
# generate_text() lives in the preflight cell above (single definition; eval-only, no training).

N = len(df_test)
PROG_EVERY = 10 if N <= 200 else 100
OUT_DIR = Path(CFG["OUTPUT_DIR"]); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_CSV = OUT_DIR / "predictions.csv"
# Resume safely: rows already recorded (e.g. after a Kaggle interruption) are kept, never redone.
rows, done = [], set()
if PRED_CSV.is_file():
    try:
        _prev = pd.read_csv(PRED_CSV)
        _need = {"sample_id", "expected_raw", "expected_norm", "predicted_raw", "predicted_norm",
                 "exact_match_raw", "exact_match", "status", "error", "fetch_s", "preprocess_s",
                 "latency_s", "num_frames", "actual_length"}
        if _need <= set(_prev.columns):
            rows = _prev.to_dict("records")
            done = set(_prev["sample_id"].astype(str))
            print(f"resuming: {len(done)} samples already recorded")
    except Exception as e:
        print(f"resume read failed ({type(e).__name__}); starting fresh")
        rows, done = [], set()

total_fetch, total_pre, total_gen = 0.0, 0.0, 0.0
t0_all = time.time()
n_new = 0
for i, r in df_test.iterrows():
    uid = str(r["uid"])
    if uid in done:
        continue
    expected_raw = str(r["text"])
    expected_norm = normalize_text(expected_raw)
    status, error, pred_raw = "ok", "", ""
    n_frames, actual_len = 0, 0
    dt_fetch = dt_pre = dt_gen = 0.0
    try:
        t0 = time.time()
        raw = POSE_STORE.read(uid)  # only this member crosses the wire
        arr543 = read_pose_to_543(raw, uid)
        n_frames = int(arr543.shape[0])
        t1 = time.time(); dt_fetch = t1 - t0
        pre, actual_len = signage_to_external_verified(arr543, use_motion=True, max_frames=500)
        t2 = time.time(); dt_pre = t2 - t1
        pt = torch.from_numpy(pre).unsqueeze(0).float()
        pred_raw = generate_text(pt, int(actual_len))
        t3 = time.time(); dt_gen = t3 - t2
        total_fetch += dt_fetch; total_pre += dt_pre; total_gen += dt_gen
    except FileNotFoundError:
        status, error = "fetch_error", "member missing"
    except Exception as e:
        status, error = "infer_error", type(e).__name__
    pred_norm = normalize_text(pred_raw)
    rows.append({"sample_id": uid, "expected_raw": expected_raw, "expected_norm": expected_norm,
                 "predicted_raw": pred_raw, "predicted_norm": pred_norm,
                 "exact_match_raw": bool(expected_raw and pred_raw == expected_raw),
                 "exact_match": bool(expected_norm and pred_norm == expected_norm),
                 "status": status, "error": error,
                 "fetch_s": round(dt_fetch, 4), "preprocess_s": round(dt_pre, 4),
                 "latency_s": round(dt_gen, 4),
                 "num_frames": n_frames, "actual_length": int(actual_len)})
    n_new += 1
    if n_new % PROG_EVERY == 0 or (i + 1) == N:
        print(f"[{i+1}/{N}] (new={n_new})")
    if n_new % int(CFG["SAVE_EVERY"]) == 0:
        pd.DataFrame(rows).to_csv(PRED_CSV, index=False)  # incremental; interruption-safe
pd.DataFrame(rows).to_csv(PRED_CSV, index=False)
total_elapsed = time.time() - t0_all
print(f"done: requested={N} new={n_new} fetch={total_fetch:.1f}s preprocess={total_pre:.1f}s inference={total_gen:.1f}s elapsed={total_elapsed:.1f}s")


In [ ]:
import json, statistics
from collections import Counter
from pathlib import Path
import pandas as pd

OUT = Path(CFG["OUTPUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
# Reporting-only bootstrap: rebuild the evaluation loop's in-memory inputs
# from the already-generated predictions.csv when this cell runs WITHOUT the
# loop above (reporting-only rerun -- no inference is repeated). In a normal
# Run All, `rows` is non-empty and this block is inert.
try:
    _mem_rows = len(rows) > 0
except NameError:
    _mem_rows = False
if not _mem_rows and (OUT / "predictions.csv").is_file():
    _disk = pd.read_csv(OUT / "predictions.csv")  # prior run's REAL predictions
    rows = _disk.to_dict("records")
    try:
        _has_dftest = len(df_test) > 0
    except NameError:
        _has_dftest = False
    if not _has_dftest:
        df_test = _disk  # length reference only (n_requested falls back to n_recorded)
    total_fetch = float(_disk["fetch_s"].sum()) if "fetch_s" in _disk.columns else 0.0
    total_pre = 0.0
    _lat_sum = float(_disk["latency_s"].sum()) if "latency_s" in _disk.columns else 0.0
    total_elapsed = total_fetch + _lat_sum
pred = pd.DataFrame(rows)  # predictions.csv already saved incrementally by the loop
okm = pred[pred["status"] == "ok"] if "status" in pred.columns else pred
n_req, n_rec = len(df_test), len(pred)
n_fetched = int((pred["status"] != "fetch_error").sum()) if "status" in pred.columns else n_rec
n_infer = len(okm)
n_ok = int(okm["exact_match"].sum()) if n_infer else 0
n_ok_raw = int(okm["exact_match_raw"].sum()) if n_infer else 0
acc = (n_ok / n_infer) if n_infer else 0.0
acc_raw = (n_ok_raw / n_infer) if n_infer else 0.0
lat = okm["latency_s"].tolist()
fet = okm["fetch_s"].tolist() if "fetch_s" in okm.columns else []
metrics = {
    "model_repo": CFG["MODEL_REPO"], "checkpoint": CFG["MODEL_VARIANT"],
    "checkpoint_dir": str(CKPT_DIR), "tokenizer": CFG["BASE_TOKENIZER"],
    "dataset": "iSign held-out test set (video-level split, seed 42)", "csv": str(CSV_PATH),
    "pose_source": POSE_KIND, "remote_chunk_mb": int(CFG["REMOTE_CHUNK_MB"]),
    "n_requested": n_req, "n_recorded": n_rec, "n_fetched_ok": n_fetched,
    "n_fetch_failed": n_rec - n_fetched, "n_inferred_ok": n_infer,
    "n_samples": n_infer, "exact_match_accuracy": acc, "exact_match_accuracy_raw": acc_raw,
    "n_correct": n_ok, "n_correct_raw": n_ok_raw, "n_incorrect": n_infer - n_ok,
    "avg_latency_s": float(sum(lat) / len(lat)) if lat else 0.0,
    "median_latency_s": float(statistics.median(lat)) if lat else 0.0,
    "total_inference_s": float(sum(lat)),
    "avg_fetch_s": float(sum(fet) / len(fet)) if fet else 0.0,
    "total_fetch_s": round(total_fetch, 3), "total_preprocessing_s": round(total_pre, 3),
    "total_elapsed_s": round(total_elapsed, 3), "device": str(DEVICE),
    "preprocessing": "verified external iSign (75 landmarks, conf 1.0, z-score axis=0, motion, pad/truncate 500)",
    "generation": {"max_length": CFG["MAX_GEN_TOKENS"], "num_beams": CFG["NUM_BEAMS"]},
    "eval_rule": "normalized: prediction == ground truth after lowercase/whitespace norm; raw: byte-exact raw equality",
    "empty_prediction_rate": float((okm['predicted_norm'] == "").mean()) if n_infer else 0.0,
}
pred.to_csv(OUT / "predictions.csv", index=False)
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
# Compact confusion: top expected->predicted pairs (sentence labels; full matrix infeasible).
pairs = Counter(zip(okm["expected_norm"], okm["predicted_norm"]))
conf = pd.DataFrame([{"expected_norm": e, "predicted_norm": p, "count": c}
                     for (e, p), c in pairs.most_common(50)])
conf.to_csv(OUT / "confusion_matrix.csv", index=False)
wrong = okm[~okm["exact_match"]]
print(f"requested={n_req} recorded={n_rec} fetched_ok={n_fetched} inferred_ok={n_infer}")
print(f"normalized acc={acc:.4f} ({n_ok}/{n_infer}) raw acc={acc_raw:.4f} ({n_ok_raw}/{n_infer})")
print(f"fetch avg={metrics['avg_fetch_s']:.3f}s inference avg={metrics['avg_latency_s']:.3f}s median={metrics['median_latency_s']:.3f}s")
print("most common incorrect predictions:")
for txt, c in Counter(wrong["predicted_norm"]).most_common(10):
    print(f"  {c}x {txt[:120]!r}")
print(f"saved: predictions.csv, metrics.json, confusion_matrix.csv -> {OUT}")
# ---------------------------------------------------------------------------
# Prediction-distribution / mode-collapse / hallucination-risk analysis.
# REPORTING ONLY: pure pandas/Counter arithmetic on recorded predictions.
# Changes nothing about the model, preprocessing, split, generation, gate,
# or criteria. Rewrites OUT/metrics.json adding analysis keys (existing keys
# untouched in meaning); predictions.csv schema unchanged.
# ---------------------------------------------------------------------------
def distribution_analysis(frame):
    """Analyze recorded predictions. Returns a JSON-serializable dict.
    ANALYSIS diagnostic only -- defines no promotion criterion and invents
    no GO threshold (see the evaluation_verdict rule below)."""
    n = int(len(frame))
    p = frame["predicted_norm"].fillna("").astype(str) if n else pd.Series([], dtype=str)
    e = frame["expected_norm"].fillna("").astype(str) if n else pd.Series([], dtype=str)
    if "exact_match" in frame.columns and n:
        ok_mask = frame["exact_match"].fillna(False).astype(bool)
    else:
        ok_mask = pd.Series([False] * n)
    pc, ec = Counter(p), Counter(e)
    top = pc.most_common(10)
    top1_txt, top1_c = (top[0] if top else ("", 0))
    top1_share = float(top1_c / n) if n else 0.0
    top5_cum = float(sum(c for _, c in top[:5]) / n) if n else 0.0
    uniq = int(len(pc))
    uratio = float(uniq / n) if n else 0.0
    sing = int(sum(1 for _, c in pc.items() if c == 1))
    n_empty = int((p == "").sum())
    ch = [len(t) for t in p]
    wd = [len(t.split()) for t in p]
    n_bad = int((~ok_mask).sum())
    wpc = Counter(p[~ok_mask])
    wtop = wpc.most_common(10)
    wt_txt, wt_c = (wtop[0] if wtop else ("", 0))
    wt_share_bad = float(wt_c / n_bad) if n_bad else 0.0
    concentrated = bool(n > 0 and (top1_share >= 0.10 or uratio <= 0.05))
    if n > 0 and int(ok_mask.sum()) == 0:
        verdict = "NO_GO"
        basis = (f"exact-match accuracy 0/{n} (normalized) on real held-out data; "
                 f"output distribution severely degenerate (dominant prediction {top1_share:.1%} "
                 f"of all samples, unique-prediction ratio {uratio:.4f})")
    else:
        verdict = "INCONCLUSIVE"
        basis = ("manual review required against repository criteria; this notebook "
                 "defines no GO threshold and invents none")
    return {
        "prediction_distribution": {
            "n_predictions": n,
            "n_unique_predictions": uniq,
            "unique_prediction_ratio": uratio,
            "top10_predictions": [
                {"prediction": t, "count": int(c), "share": float(c / n) if n else 0.0}
                for t, c in top
            ],
            "top1_prediction": top1_txt,
            "top1_count": int(top1_c),
            "top1_share": top1_share,
            "top5_cumulative_share": top5_cum,
            "singleton_predictions": sing,
            "singleton_share": float(sing / n) if n else 0.0,
            "n_empty_predictions": n_empty,
            "empty_prediction_rate": float(n_empty / n) if n else 0.0,
            "prediction_char_length": {
                "mean": float(sum(ch) / n) if n else 0.0,
                "median": float(statistics.median(ch)) if ch else 0.0,
                "min": int(min(ch)) if ch else 0,
                "max": int(max(ch)) if ch else 0,
            },
            "prediction_word_count": {
                "mean": float(sum(wd) / n) if n else 0.0,
                "median": float(statistics.median(wd)) if wd else 0.0,
                "min": int(min(wd)) if wd else 0,
                "max": int(max(wd)) if wd else 0,
            },
            "n_unique_expected_labels": int(len(ec)),
            "top10_expected_labels": [
                {"expected": t, "count": int(c)} for t, c in ec.most_common(10)
            ],
        },
        "mode_collapse": {
            "note": "ANALYSIS diagnostic only -- describes output concentration; it is NOT an official promotion criterion (the repository defines no cutoff for this candidate) and changes no gate.",
            "dominant_prediction": top1_txt,
            "dominant_count": int(top1_c),
            "dominant_share": top1_share,
            "top5_cumulative_share": top5_cum,
            "unique_prediction_ratio": uratio,
            "highly_concentrated": concentrated,
            "concentration_rule": "descriptive heuristic for this analysis line only: top1_share >= 10% OR unique_ratio <= 5%",
        },
        "hallucination_risk": {
            "note": "Wrong predictions are NOT automatically hallucinations; this section reports repetition/fluency patterns that warrant the controlled negative tests in docs/EXTERNAL_EVALUATION.md.",
            "random_noise_probe": "not performed in this notebook (execution smoke tests used synthetic input only and make no quality claim); empty-frame / random-movement negative tests belong to the separate webcam protocol",
            "n_incorrect": n_bad,
            "top_wrong_prediction": wt_txt,
            "top_wrong_count": int(wt_c),
            "top_wrong_share_of_incorrect": wt_share_bad,
            "top10_wrong_predictions": [
                {"prediction": t, "count": int(c)} for t, c in wtop
            ],
            "empty_output_count": n_empty,
            "empty_output_rate": float(n_empty / n) if n else 0.0,
        },
        "evaluation_verdict": verdict,
        "evaluation_verdict_basis": basis,
        "evaluation_verdict_scope": "Candidate-level evaluation verdict for t5-small-motion recorded in this notebook only; does NOT change SIGNSAGE_MODEL_PROMOTED, the production registry, or any deployment gate.",
    }

_analysis = distribution_analysis(okm)
metrics.update(_analysis)
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"unique predictions={_analysis['prediction_distribution']['n_unique_predictions']}/{n_infer} (ratio={_analysis['prediction_distribution']['unique_prediction_ratio']:.4f})")
print(f"dominant={_analysis['mode_collapse']['dominant_count']}x ({_analysis['mode_collapse']['dominant_share']:.1%}) top5-cum={_analysis['mode_collapse']['top5_cumulative_share']:.1%} concentrated={_analysis['mode_collapse']['highly_concentrated']}")
print(f"verdict={metrics['evaluation_verdict']} :: {metrics['evaluation_verdict_basis']}")
print(f"updated metrics.json with distribution + verdict -> {OUT}")


In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

OUT = Path(globals().get("CFG", {}).get("OUTPUT_DIR", "/kaggle/working/evaluation_output"))
OUT.mkdir(parents=True, exist_ok=True)
# Reporting-only fallback: rebuild report inputs from the already-generated
# artifacts when this cell runs WITHOUT the evaluation cells above (no
# inference is repeated). In a normal Run All this block is inert. If the
# on-disk metrics.json predates the distribution analysis, run the metrics
# cell first (reporting-only, no inference) -- it adds the analysis keys.
try:
    _mem_ok = isinstance(metrics, dict) and len(pred) > 0 and len(okm) > 0
except NameError:
    _mem_ok = False
if not _mem_ok:
    pred = pd.read_csv(OUT / "predictions.csv")  # fails loudly if no prior run exists
    okm = pred[pred["status"] == "ok"] if "status" in pred.columns else pred
    try:
        metrics = json.loads((OUT / "metrics.json").read_text())
    except (OSError, ValueError):
        metrics = {}
    if "prediction_distribution" not in metrics:
        raise RuntimeError("metrics.json predates the distribution analysis -- run the metrics cell first (reporting-only, no inference), then rerun this cell.")
_csv = globals().get("CSV_PATH", metrics.get("csv", "?"))
_pose_kind = globals().get("POSE_KIND", metrics.get("pose_source", "?"))
_pd = metrics.get("prediction_distribution", {})
_mc = metrics.get("mode_collapse", {})
_hr = metrics.get("hallucination_risk", {})
wrong = okm[~okm["exact_match"]] if "exact_match" in okm.columns else okm.iloc[0:0]
top_wrong = Counter(wrong["predicted_norm"].fillna("").astype(str)).most_common(10)
top_pairs = Counter(zip(okm["expected_norm"].fillna("").astype(str), okm["predicted_norm"].fillna("").astype(str))).most_common(10)
total_mb = sum(p.stat().st_size for p in OUT.glob("*") if p.is_file()) / 1e6
report = f"""# Evaluation report — external iSign t5-small-motion (dataset-level, real data)

Evaluation verdict: **{metrics.get('evaluation_verdict', 'INCONCLUSIVE')}** — {metrics.get('evaluation_verdict_basis', 'no verdict basis recorded')}
(Verdict scope: candidate-level evaluation verdict recorded in this notebook only; it does NOT change `SIGNSAGE_MODEL_PROMOTED`, the production registry, or any deployment gate.)

## 1. Model
- Repository: `{metrics.get('model_repo', '?')}` — checkpoint `{metrics.get('checkpoint', '?')}` ONLY (pose_dim=600, motion=True).
- Architecture: PoseEncoder MLP (Linear 600→256 → ReLU → Dropout 0.1 → Linear 256→512) + T5-small decoder.
- Tokenizer: `{metrics.get('tokenizer', '?')}`. Generation: max_length={metrics.get('generation', {}).get('max_length', '?')}, beams={metrics.get('generation', {}).get('num_beams', '?')}, early_stopping=True, length_penalty=2.0.
- Checkpoint location this run: `{metrics.get('checkpoint_dir', '?')}`.
- No training, no fine-tuning, no weight changes anywhere in this notebook (`model.eval()` + `torch.no_grad()` only).

## 2. Dataset and held-out split
- Source: iSign held-out test split (video-level 80/10/10, seed 42) from `{_csv}`; full target 1887.
- Requested: {metrics.get('n_requested', '?')} | recorded: {metrics.get('n_recorded', '?')} | fetched OK: {metrics.get('n_fetched_ok', '?')} | fetch failed: {metrics.get('n_fetch_failed', '?')} | inferred OK: {metrics.get('n_inferred_ok', '?')}.
- Pose access: authenticated HTTP Range requests against the 4-part ZIP (ONE ZIP64 archive); only evaluated members retrieved, archive never downloaded (source: {_pose_kind}, chunk {metrics.get('remote_chunk_mb', '?')} MB).
- Production pipeline untouched; registry untouched; `SIGNSAGE_MODEL_PROMOTED` untouched (no promotion).

## 3. Preprocessing contract (frozen, verified)
- Verified external contract ONLY: 75 landmarks (POSE 33 + LEFT_HAND 21 + RIGHT_HAND 21; FACE discarded), confidence=1.0 fallback → 300 values/frame, z-score across axis=0 BEFORE motion, velocity[0]=0 + velocity[t]=pose[t]-pose[t-1], concat [pose|velocity] → 600/frame, pad/truncate to exactly 500 frames. No SignSage hip-centering, no preprocessing changes.
- Eval rule: normalized exact-match (prediction == ground truth after lowercase/whitespace norm) counts as correct; raw byte-exact equality reported alongside. Fluent English output was NEVER counted as correct unless it exactly matched ground truth. Near-paraphrases count as incorrect by design (no silent leniency).

## 4. Hardware and timing
- Device: {metrics.get('device', '?')}.
- Avg fetch: {metrics.get('avg_fetch_s', 0):.3f}s | avg inference: {metrics.get('avg_latency_s', 0):.3f}s | median inference: {metrics.get('median_latency_s', 0):.3f}s.
- Total fetch: {metrics.get('total_fetch_s', 0):.1f}s | total inference: {metrics.get('total_inference_s', 0):.1f}s | elapsed: {metrics.get('total_elapsed_s', 0):.1f}s.

## 5. Exact-match results
- Normalized: {metrics.get('exact_match_accuracy', 0):.4f} ({metrics.get('n_correct', 0)}/{metrics.get('n_inferred_ok', 0)}).
- Raw: {metrics.get('exact_match_accuracy_raw', 0):.4f} ({metrics.get('n_correct_raw', 0)}/{metrics.get('n_inferred_ok', 0)}).
- Incorrect: {metrics.get('n_incorrect', 0)}.

## 6. Prediction distribution
- Unique predictions: {_pd.get('n_unique_predictions', '?')} / {metrics.get('n_inferred_ok', '?')} (ratio {_pd.get('unique_prediction_ratio', 0):.4f}).
- Top-1 prediction covers {_pd.get('top1_share', 0):.1%} of all samples ({_pd.get('top1_count', 0)}x: {_pd.get('top1_prediction', '')!r}).
- Top-5 predictions cover {_pd.get('top5_cumulative_share', 0):.1%} of all samples.
- Singleton predictions (occur exactly once): {_pd.get('singleton_predictions', '?')} ({_pd.get('singleton_share', 0):.1%}).
- Empty predictions: {_pd.get('n_empty_predictions', '?')} (rate {_pd.get('empty_prediction_rate', 0):.4f}).
- Prediction length (chars): mean {_pd.get('prediction_char_length', {}).get('mean', 0):.1f} | median {_pd.get('prediction_char_length', {}).get('median', 0):.1f} | min {_pd.get('prediction_char_length', {}).get('min', 0)} | max {_pd.get('prediction_char_length', {}).get('max', 0)}.
- Prediction length (words): mean {_pd.get('prediction_word_count', {}).get('mean', 0):.1f} | median {_pd.get('prediction_word_count', {}).get('median', 0):.1f} | min {_pd.get('prediction_word_count', {}).get('min', 0)} | max {_pd.get('prediction_word_count', {}).get('max', 0)}.
- Unique expected labels: {_pd.get('n_unique_expected_labels', '?')}.

Top-10 predictions:
"""
for _it in _pd.get('top10_predictions', []):
    report += f"- {_it['count']}x ({_it['share']:.1%}): {_it['prediction'][:160]!r}\n"
report += "\nTop-10 expected labels:\n"
for _it in _pd.get('top10_expected_labels', []):
    report += f"- {_it['count']}x: {_it['expected'][:160]!r}\n"
report += f"""
## 7. Mode-collapse diagnostic (ANALYSIS ONLY)
- {_mc.get('note', '')}
- Dominant prediction: {_mc.get('dominant_count', '?')}x ({_mc.get('dominant_share', 0):.1%}): {_mc.get('dominant_prediction', '')!r}.
- Top-5 cumulative share: {_mc.get('top5_cumulative_share', 0):.1%}.
- Unique-prediction ratio: {_mc.get('unique_prediction_ratio', 0):.4f}.
- Highly concentrated: {_mc.get('highly_concentrated', '?')} ({_mc.get('concentration_rule', '')}).

## 8. Hallucination-risk observations
- {_hr.get('note', '')}
- Random/noise probe: {_hr.get('random_noise_probe', '')}
- Incorrect samples: {_hr.get('n_incorrect', '?')}; top wrong prediction covers {_hr.get('top_wrong_share_of_incorrect', 0):.1%} of them ({_hr.get('top_wrong_count', 0)}x: {_hr.get('top_wrong_prediction', '')!r}).
- Empty-output count: {_hr.get('empty_output_count', '?')} (rate {_hr.get('empty_output_rate', 0):.4f}); empty output is a correct rejection, not a translation.

Most common incorrect predictions:
"""
for txt, c in top_wrong:
    report += f"- incorrect prediction {c}x: {txt[:160]!r}\n"
report += "\nTop expected->predicted pairs:\n"
for (e, p), c in top_pairs:
    report += f"- {c}x expected={e[:100]!r} predicted={p[:100]!r}\n"
report += f"""
## 9. Limitations
- Sentence-level labels: no word-level confusion matrix; pairs above are the compact substitute.
- Confidence fallback 1.0 (no per-landmark confidence in this pipeline; training distribution unknown).
- Studio training data vs any webcam distribution shift is NOT measured here.
- Held-out split reconstructed via video-level policy (seed 42); if the publisher's exact 1,887 IDs differ, counts shift -- actual n is recorded above.
- Accuracy is exact-match only; near-paraphrases count as incorrect by design (no silent leniency).
- Remote member fetch can fail transiently (network/CDN); failures are recorded per-sample as fetch_error and never silently retried into invented predictions. predictions.csv is written incrementally so interruptions lose nothing.
- Chunk granularity ({metrics.get('remote_chunk_mb', '?')} MB) bounds per-member transfer waste.
- Verdict scope: {metrics.get('evaluation_verdict_scope', '')}

## 10. Evaluation conclusion
- Real-data evaluation completed successfully (preflight PASS; {metrics.get('n_inferred_ok', 0)}/{metrics.get('n_requested', 0)} requested samples fetched and inferred).
- Exact-match accuracy was {metrics.get('n_correct', 0)}/{metrics.get('n_inferred_ok', 0)} (normalized {metrics.get('exact_match_accuracy', 0):.4f}; raw {metrics.get('n_correct_raw', 0)}/{metrics.get('n_inferred_ok', 0)}).
- The model therefore does NOT demonstrate usable recognition quality on this evaluation set.
- Repeated fluent incorrect outputs (dominant {_mc.get('dominant_count', '?')}x at {_mc.get('dominant_share', 0):.1%}, top-5 cumulative {_mc.get('top5_cumulative_share', 0):.1%}) indicate severe prediction concentration / failure behavior.
- This notebook does NOT authorize production deployment or model promotion.

Artifacts: predictions.csv, metrics.json, confusion_matrix.csv, evaluation_report.md (total ~{total_mb:.2f} MB).
"""
(OUT / "evaluation_report.md").write_text(report)
print(f"wrote evaluation_report.md ({total_mb:.2f} MB total in {OUT})")
print("== Final config ==")
print(f"Model: {metrics.get('checkpoint', '?')} | Dataset: iSign held-out test | Samples: {metrics['n_inferred_ok']} | Preprocessing: verified external | Device: {metrics['device']}")
print("== Calibration summary ==")
print(f"REMOTE ACCESS: {'PASS' if _pose_kind == 'remote' and globals().get('PREFLIGHT_OK', False) else ('LOCAL (' + str(_pose_kind) + ')' if _pose_kind in ('dir', 'zip') else 'FAIL')}")
print(f"CSV ACCESS: {'PASS' if globals().get('CSV_OK', False) else 'FAIL'}")
print(f"POSE MEMBER PREFLIGHT: {'PASS' if globals().get('PREFLIGHT_OK', False) else 'FAIL'}")
print(f"MODEL LOAD: {'PASS' if globals().get('MODEL_OK', False) else 'FAIL'}")
print(f"SINGLE SAMPLE INFERENCE: {'PASS' if globals().get('SINGLE_INFER_OK', False) else 'FAIL'}")
print(f"EVALUATION SAMPLES: {metrics['n_requested']}")
print(f"SUCCESSFUL SAMPLES: {metrics['n_inferred_ok']}")
print(f"FAILED SAMPLES: {metrics['n_recorded'] - metrics['n_inferred_ok']}")
print(f"TOTAL TIME: {metrics['total_elapsed_s']:.1f}s")
print(f"AVG FETCH TIME: {metrics['avg_fetch_s']:.3f}s")
print(f"AVG INFERENCE TIME: {metrics['avg_latency_s']:.3f}s")
print(f"EVALUATION VERDICT: {metrics.get('evaluation_verdict', 'INCONCLUSIVE')} :: {metrics.get('evaluation_verdict_basis', '')}")
print(f"UNIQUE PREDICTIONS: {_pd.get('n_unique_predictions', '?')} (ratio {_pd.get('unique_prediction_ratio', 0):.4f}) | DOMINANT SHARE: {_mc.get('dominant_share', 0):.1%} | TOP5 CUM: {_mc.get('top5_cumulative_share', 0):.1%}")
